In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Reshape, Flatten, Conv2D, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import initializers

In [ ]:
# Load the Fashion-MNIST dataset
(x_train, _), (_, _) = fashion_mnist.load_data()

In [ ]:
# Normalize the images to [0, 1]
x_train = x_train / 255.0
x_train = np.expand_dims(x_train, axis=-1)  # Add channel dimension (28, 28, 1)

In [ ]:
# Set the dimensions of the latent space (input to the generator)
latent_dim = 100

In [ ]:
# Build the generator model
def build_generator():
    model = Sequential()
    model.add(Dense(128, input_dim=latent_dim, kernel_initializer=initializers.RandomNormal(stddev=0.02)))
    model.add(LeakyReLU(0.2))
    model.add(Reshape((7, 7, 128)))
    model.add(Conv2DTranspose(128, (4, 4), strides=2, padding='same', kernel_initializer=initializers.RandomNormal(stddev=0.02)))
    model.add(LeakyReLU(0.2))
    model.add(Conv2DTranspose(1, (4, 4), strides=2, padding='same', activation='tanh', kernel_initializer=initializers.RandomNormal(stddev=0.02)))
    return model

In [ ]:
# Build the discriminator model
def build_discriminator():
    model = Sequential()
    model.add(Conv2D(64, (3, 3), strides=2, padding='same', input_shape=(28, 28, 1)))
    model.add(LeakyReLU(0.2))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    return model

In [ ]:
# Build the GAN model (generator + discriminator)
def build_gan(generator, discriminator):
    discriminator.trainable = False  # Freeze the discriminator during generator training
    model = Sequential([generator, discriminator])
    return model

In [ ]:
# Compile the discriminator and the GAN
discriminator = build_discriminator()
discriminator.compile(optimizer=Adam(learning_rate=0.0002, beta_1=0.5), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
generator = build_generator()

In [ ]:
gan = build_gan(generator, discriminator)
gan.compile(optimizer=Adam(learning_rate=0.0002, beta_1=0.5), loss='binary_crossentropy')

In [ ]:
# Training parameters
epochs = 50
batch_size = 64
half_batch = batch_size // 2

In [ ]:
# Training the GAN
def train_gan(epochs, batch_size):
    for epoch in range(epochs):

In [ ]:
        # Train the discriminator
        idx = np.random.randint(0, x_train.shape[0], half_batch)
        real_images = x_train[idx]
        real_labels = np.ones((half_batch, 1))

In [ ]:
        noise = np.random.normal(0, 1, (half_batch, latent_dim))
        fake_images = generator.predict(noise)
        fake_labels = np.zeros((half_batch, 1))

In [ ]:
        # Train the discriminator on real and fake images
        d_loss_real = discriminator.train_on_batch(real_images, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)


In [ ]:
        # Train the generator
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        valid_labels = np.ones((batch_size, 1))
        g_loss = gan.train_on_batch(noise, valid_labels)


In [ ]:
        # Print progress
        if epoch % 100 == 0:
            print(f"{epoch} [D loss: {d_loss[0]} | D accuracy: {100*d_loss[1]}] [G loss: {g_loss}]")

In [ ]:
        # Save generated images at intervals
        if epoch % 1000 == 0:
            save_generated_images(epoch)

In [ ]:
# Function to save and display generated images
def save_generated_images(epoch, examples=10, dim=(1, 10), figsize=(10, 1)):
    noise = np.random.normal(0, 1, (examples, latent_dim))
    generated_images = generator.predict(noise)
    generated_images = 0.5 * generated_images + 0.5  # Rescale images to [0, 1]

In [ ]:
    fig, axs = plt.subplots(dim[0], dim[1], figsize=figsize)
    cnt = 0
    for i in range(dim[0]):
        for j in range(dim[1]):
            axs[i, j].imshow(generated_images[cnt, :, :, 0], cmap='gray')
            axs[i, j].axis('off')
            cnt += 1
    plt.tight_layout()
    plt.savefig(f"gan_generated_image_epoch_{epoch}.png")
    plt.close()

In [ ]:
# Train the GAN and save images at intervals
train_gan(epochs, batch_size)